# Early Detection of ADHD in the Nigerian Population Using AI

**Master's Capstone Research Project** — ADHD Screening & Referral Support Tool (Non-Diagnostic)

Target population: Nigerian pediatric demographics, ages 4–15. Python 3.10+ / scikit-learn.

> This is a **screening and referral support tool**, not an automated diagnostic tool. It assists overloaded pediatricians by triaging children before full clinical consultations.

## 1. Background & Clinical Motivation

Attention-Deficit/Hyperactivity Disorder (ADHD) is one of the most prevalent childhood neurodevelopmental disorders globally. In Nigeria, clinical diagnosis is often delayed due to:

- High patient-to-psychiatrist ratios in public hospitals.
- Sociocultural stigma surrounding mental health.
- Lack of accessible, localized screening questionnaires.
- Limited exposure of teachers and primary health workers to ADHD symptom recognition.

## 2. Project Scope & Objectives

1. Parse standardized parent/teacher ADHD questionnaire responses (DSM-5 aligned).
2. Use machine learning to assess overall risk level (Low / Moderate / High).
3. Connect at-risk profiles with localized Nigerian medical pathways.
4. Provide structured, transparent risk assessments that clinical workers can read easily.

## 3. Questionnaire Mapping Schema (5-point Likert)

| Response Label | Score |
|---|---|
| Never | 0 |
| Rarely | 1 |
| Sometimes | 2 |
| Often | 3 |
| Very Often | 4 |

## Step 1 — Import Libraries

Core data, plotting, and scikit-learn ML imports.

In [ ]:
import pandas as pd                # Structured data manipulation
import numpy as np                 # Mathematical vector calculations
import matplotlib.pyplot as plt    # Core plotting library
import seaborn as sns              # Statistical visualization
import joblib                      # Saving/loading trained ML model files

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries imported successfully!')

## Step 2 — Create Sample Dataset

Clinical datasets are heavily guarded by ethical constraints, so we simulate 500 records representing pediatric screening surveys in Nigerian communities. We deliberately inject missing values (15 missing `Age`, 12 missing `Fidgeting`) so we can demonstrate realistic clinical data cleaning in Step 3.

In [ ]:
rng = np.random.default_rng(42)
n_samples = 500

# Age (4-15), Gender (Male 55% / Female 45%), School Level (mapped from age)
ages = rng.integers(4, 16, n_samples)
gender = rng.choice(['Male', 'Female'], n_samples, p=[0.55, 0.45])
school_levels = [
    'Nursery' if a <= 5 else 'Primary' if a <= 11 else 'Junior Secondary'
    for a in ages
]

# Multicenter Nigerian state representativeness
nigerian_states = rng.choice(
    ['Lagos', 'Kano', 'Oyo', 'Enugu', 'Rivers', 'Kaduna', 'Plateau'],
    size=n_samples,
    p=[0.25, 0.15, 0.15, 0.12, 0.13, 0.10, 0.10],
)

# Two latent traits drive the six DSM-5-aligned symptom scores.
latent_hyper = rng.binomial(4, 0.42, n_samples)
latent_inatt = rng.binomial(4, 0.38, n_samples)
noise = lambda: rng.integers(-1, 2, n_samples)
clip = lambda a: np.clip(a, 0, 4).astype(int)

fidgeting        = clip(latent_hyper + noise())
staying_seated   = clip(latent_hyper + noise())
interrupts       = clip(latent_hyper + noise())
distracted       = clip(latent_inatt + noise())
completing_tasks = clip(latent_inatt + noise())
loses_items      = clip(latent_inatt + noise())

# Hyperactivity Score (max = 24)
hyperactivity_score = (fidgeting + staying_seated + distracted +
                       interrupts + completing_tasks + loses_items)

# Risk labels: Low (<8), Moderate (8-14), High (>=15)
risk_labels = [
    'High Risk' if s >= 15 else 'Moderate Risk' if s >= 8 else 'Low Risk'
    for s in hyperactivity_score
]

raw_df = pd.DataFrame({
    'Age': ages.astype(float),
    'Gender': gender,
    'School_Level': school_levels,
    'State': nigerian_states,
    'Fidgeting': fidgeting.astype(float),
    'Staying_Seated': staying_seated,
    'Distracted': distracted,
    'Interrupts': interrupts,
    'Completing_Tasks': completing_tasks,
    'Loses_Items': loses_items,
    'Hyperactivity_Score': hyperactivity_score,
    'ADHD_Risk': risk_labels,
})

# Inject missing values to simulate incomplete clinical responses
missing_age_idx  = rng.choice(n_samples, size=15, replace=False)
missing_fidg_idx = rng.choice(n_samples, size=12, replace=False)
raw_df.loc[missing_age_idx,  'Age']       = np.nan
raw_df.loc[missing_fidg_idx, 'Fidgeting'] = np.nan

print(f'Shape: {raw_df.shape}')
print(f"Missing Age      : {raw_df['Age'].isna().sum()}")
print(f"Missing Fidgeting: {raw_df['Fidgeting'].isna().sum()}")
raw_df.head()

## Step 3 — Data Preprocessing

### 3.1 Handle missing values
- **Age** imputed with the **median** (robust to outliers).
- **Fidgeting** imputed with the **mode** (most frequent discrete Likert answer).

### 3.2 Encode categorical variables

| Variable | Encoding | Values |
|---|---|---|
| Gender | Binary | Male=1, Female=0 |
| School_Level | Ordinal | Nursery=0, Primary=1, Junior Secondary=2 |
| ADHD_Risk | Ordinal | Low=0, Moderate=1, High=2 |

In [ ]:
clean_df = raw_df.copy()

clean_df['Age']       = clean_df['Age'].fillna(clean_df['Age'].median())
clean_df['Fidgeting'] = clean_df['Fidgeting'].fillna(clean_df['Fidgeting'].mode().iloc[0])

symptom_cols = ['Fidgeting','Staying_Seated','Distracted',
                'Interrupts','Completing_Tasks','Loses_Items']
clean_df['Hyperactivity_Score'] = clean_df[symptom_cols].sum(axis=1).astype(int)
clean_df['ADHD_Risk'] = clean_df['Hyperactivity_Score'].apply(
    lambda s: 'High Risk' if s >= 15 else 'Moderate Risk' if s >= 8 else 'Low Risk'
)

gender_map = {'Male': 1, 'Female': 0}
school_map = {'Nursery': 0, 'Primary': 1, 'Junior Secondary': 2}
risk_map   = {'Low Risk': 0, 'Moderate Risk': 1, 'High Risk': 2}

clean_df['Gender']         = clean_df['Gender'].map(gender_map).astype(int)
clean_df['School_Level']   = clean_df['School_Level'].map(school_map).astype(int)
clean_df['ADHD_Risk_Code'] = clean_df['ADHD_Risk'].map(risk_map).astype(int)
clean_df['Age']            = clean_df['Age'].astype(int)
for c in symptom_cols:
    clean_df[c] = clean_df[c].astype(int)

print('Missing values remaining:', clean_df.isna().sum().sum())
clean_df.head()

### 3.3 Train-Test Split

80% training / 20% testing. `stratify=y` keeps both subsets at the same Low/Moderate/High ratio.

In [ ]:
feature_cols = ['Age', 'Gender', 'School_Level', *symptom_cols]
X = clean_df[feature_cols]
y = clean_df['ADHD_Risk_Code']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}   Test: {X_test.shape}')
print('Train class balance:')
print(y_train.map({0:'Low',1:'Mod',2:'High'}).value_counts())

## Step 4 — Exploratory Data Analysis (EDA)

Three visualizations check demographic bias and clinical trends:
1. ADHD Risk Distribution (countplot — Low / Moderate / High)
2. Hyperactivity Score distribution by Gender (boxplot)
3. Screening Outcomes by School Level (grouped countplot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
risk_order = ['Low Risk', 'Moderate Risk', 'High Risk']
risk_palette = ['#22c55e', '#eab308', '#ef4444']

sns.countplot(
    data=clean_df, x='ADHD_Risk', ax=axes[0],
    order=risk_order, hue='ADHD_Risk',
    palette=risk_palette, legend=False,
)
axes[0].set_title('ADHD Risk Distribution')

gender_lab = clean_df.assign(GenderLabel=clean_df['Gender'].map({1:'Male',0:'Female'}))
sns.boxplot(
    data=gender_lab, x='GenderLabel', y='Hyperactivity_Score',
    ax=axes[1], hue='GenderLabel',
    palette=['#3b82f6', '#f43f5e'], legend=False,
)
axes[1].set_title('Hyperactivity Score by Gender')
axes[1].set_xlabel('Gender')

school_lab = clean_df.assign(SchoolLabel=clean_df['School_Level'].map({0:'Nursery',1:'Primary',2:'Junior Secondary'}))
sns.countplot(
    data=school_lab, x='SchoolLabel', hue='ADHD_Risk',
    ax=axes[2], hue_order=risk_order,
    palette=risk_palette,
    order=['Nursery','Primary','Junior Secondary'],
)
axes[2].set_title('Screening Outcomes by School Level')
axes[2].set_xlabel('School Level')

plt.tight_layout(); plt.show()

## Step 5 — Train Machine Learning Models

| Model | Hyperparameters | Purpose |
|---|---|---|
| Logistic Regression | `max_iter=1000` (multinomial by default in sklearn ≥1.7) | Linear probabilistic baseline |
| Decision Tree | `max_depth=5`, `min_samples_split=10` | Explainable threshold model |
| Random Forest | `n_estimators=100`, `max_depth=6` | Ensemble bagging — best generalization |

In [ ]:
# sklearn >=1.7 removed the explicit `multi_class` kwarg; multinomial is now default.
log_reg     = LogisticRegression(max_iter=1000, random_state=42)
dec_tree    = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
rand_forest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

log_reg.fit(X_train, y_train)
dec_tree.fit(X_train, y_train)
rand_forest.fit(X_train, y_train)
print('All three models trained.')

## Step 6 — Evaluate & Compare Models

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    return {
        'model':     name,
        'accuracy':  accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_test, pred, average='weighted', zero_division=0),
        'pred':      pred,
    }

results = [
    evaluate('Logistic Regression', log_reg),
    evaluate('Decision Tree',       dec_tree),
    evaluate('Random Forest',       rand_forest),
]
pd.DataFrame([{k:v for k,v in r.items() if k != 'pred'} for r in results])

In [ ]:
metrics = ['accuracy','precision','recall','f1']
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results)); width = 0.2
for i, m in enumerate(metrics):
    ax.bar(x + i*width, [r[m] for r in results], width, label=m.capitalize())
ax.set_xticks(x + width*1.5)
ax.set_xticklabels([r['model'] for r in results])
ax.set_ylim(0, 1.05); ax.set_ylabel('Weighted score')
ax.set_title('Model Comparison'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ['Low Risk', 'Moderate Risk', 'High Risk']
for ax, r in zip(axes, results):
    cm = confusion_matrix(y_test, r['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(r['model']); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
importances = pd.Series(rand_forest.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances.index, importances.values, color='#2563eb')
ax.set_title('Random Forest Feature Importance'); ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

In [ ]:
print('Random Forest classification report:\n')
print(classification_report(y_test, rand_forest.predict(X_test),
                            target_names=labels, zero_division=0))

## Step 7 — Save Best Model with joblib

Saving the trained model lets clinical mobile apps load and run predictions offline — important in Nigerian states with limited connectivity.

In [ ]:
from pathlib import Path
models_dir = Path('..') / 'models'
models_dir.mkdir(exist_ok=True)

joblib.dump(rand_forest, models_dir / 'adhd_screening_rf_model.joblib')

meta_config = {
    'features':         feature_cols,
    'symptom_features': symptom_cols,
    'risk_mapping':     {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk'},
    'gender_mapping':   gender_map,
    'school_mapping':   school_map,
    'thresholds':       {'low_max_exclusive': 8, 'moderate_max_inclusive': 14, 'high_min': 15},
}
joblib.dump(meta_config, models_dir / 'adhd_model_metadata.joblib')
print('Saved:')
print('  ', models_dir / 'adhd_screening_rf_model.joblib')
print('  ', models_dir / 'adhd_model_metadata.joblib')

## Step 8 — Prediction & Clinical Referral Pipeline

`predict_adhd_screening_risk()` loads the saved model, processes survey inputs, runs inference, and prints localized Nigerian clinical referrals.

In [ ]:
REFERRALS = {
    'High Risk': [
        'Child & Adolescent Psychiatry Department, LUTH (Idi-Araba, Lagos)',
        'Pediatric/Child Mental Health Unit, UCH (Ibadan, Oyo State)',
        'Federal Neuro-Psychiatric Hospital, Yaba (Lagos)',
        'Child Psychiatry Clinic, closest regional State Teaching Hospital',
        'School Action: Recommend IEP (Individualized Education Program) adjustments',
    ],
    'Moderate Risk': [
        'Consult with school teachers to monitor focus across classroom settings.',
        'Recommend a physical assessment (vision & hearing) to rule out sensory delays.',
        'Schedule a follow-up screening questionnaire in 3-6 months.',
    ],
    'Low Risk': [
        'Symptom load falls within expected bounds of standard pediatric development.',
        'Provide standard healthy lifestyle and focus behaviors guidance.',
    ],
}

def predict_adhd_screening_risk(age, gender, school_level,
                                fidgeting, staying_seated, distracted,
                                interrupts, completing_tasks, loses_items):
    model = joblib.load(models_dir / 'adhd_screening_rf_model.joblib')
    meta  = joblib.load(models_dir / 'adhd_model_metadata.joblib')

    gender_enc = meta['gender_mapping'].get(gender, 0)
    school_enc = meta['school_mapping'].get(school_level, 1)

    row = {
        'Age': age, 'Gender': gender_enc, 'School_Level': school_enc,
        'Fidgeting': fidgeting, 'Staying_Seated': staying_seated,
        'Distracted': distracted, 'Interrupts': interrupts,
        'Completing_Tasks': completing_tasks, 'Loses_Items': loses_items,
    }
    X = pd.DataFrame([row])[meta['features']]

    predicted_class_id = int(model.predict(X)[0])
    risk_level = meta['risk_mapping'].get(predicted_class_id, 'Unknown')
    total = fidgeting + staying_seated + distracted + interrupts + completing_tasks + loses_items

    print('='*70)
    print(' ADHD Screening Report (Non-Diagnostic)')
    print('='*70)
    print(f' Age              : {age}')
    print(f' Gender           : {gender}')
    print(f' School Level     : {school_level}')
    print(f' Hyperactivity    : {total} / 24')
    print(f' Predicted Risk   : {risk_level}')
    print()
    print(' Recommended Action / Referrals:')
    for line in REFERRALS.get(risk_level, []):
        print(f'   - {line}')
    print()
    print(' NOTE: This is a screening tool, NOT a diagnosis.')
    print('='*70)
    return risk_level, total

## Step 9 — Final Prediction Example (Capstone Test Case)

| Feature | Value | Likert Label |
|---|---|---|
| Age | 8 | — |
| Gender | Male | — |
| School Level | Primary | — |
| Fidgeting | 3 | Often |
| Difficulty Remaining Seated | 2 | Sometimes |
| Easily Distracted | 4 | Very Often |
| Interrupts Others | 3 | Often |
| Difficulty Completing Tasks | 4 | Very Often |
| Loses Items | 2 | Sometimes |

In [ ]:
risk, score = predict_adhd_screening_risk(
    age=8, gender='Male', school_level='Primary',
    fidgeting=3, staying_seated=2, distracted=4,
    interrupts=3, completing_tasks=4, loses_items=2,
)
print(f'\nReturned: risk={risk!r}, score={score}')

## Capstone Defense Guidance

**Model performance overview**

| Model | Strengths | Weaknesses |
|---|---|---|
| Random Forest ✓ | Highest generalizability; limits overfitting via bagging; best for discrete threshold features | Less interpretable than a single tree |
| Logistic Regression | Clinically transparent; extractable odds ratios | Struggles with non-linear boundaries |
| Decision Tree | Highly explainable to clinicians | Prone to overfitting on narrow synthetic nodes |

**Key academic takeaways**
1. **Clinical scoping** — always state this is a screening tool, NOT a diagnosis.
2. **Missing data pipeline** — Age (median) and Fidgeting (mode) imputation prove understanding of real-world clinical data noise.
3. **Clinical bias & ethical AI** — boys are clinically weighted more in ADHD prevalence; thresholds must be calibrated to avoid missing the inattentive-type presentation more common in girls.
4. **Localization** — saving to `.joblib` lets clinical nurses run the tool inside a mobile app across Nigerian states with intermittent connectivity.

*End of Capstone Notebook — ADHD Screening & Referral Support Tool, Nigerian Pediatric Context.*